In [50]:
import numpy as np
import torch

# Experiment inputs and outputs
DEPTH_LEVELS = [
    2.5,
    10.0,
    22.5,
    40.0,
    65.0,
    105.0,
    165.0,
    250.0,
    375.0,
    550.0,
    775.0,
    1050.0,
    1400.0,
    1850.0,
    2400.0,
    3100.0,
    4000.0,
    5000.0,
    6000.0
    ]

DEPTH_I_LEVELS = [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10",
    "11",
    "12",
    "13",
    "14",
    "15",
    "16",
    "17",
    "18"
]

INPT_VARS = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "ur", "vr"],
    "3": ["um", "vm", "Tm"],
    "4": ["um", "vm", "ur", "vr", "Tm", "Tr"],
    "5": ["ur", "vr"],
    "6": ["ur", "vr", "Tr"],
    "7": ["Tm"],
    "8": ["Tm", "Tr"],
    "9": ["u", "v"],
    "10": ["u", "v", "T"],
    "11": ["tau_u", "tau_v"],
    "12": ["tau_u", "tau_v", "t_ref"],
    "3D": ["uo", "vo", "thetao", "so", "zos"],
    "2D": [
        k + DEPTH_I_LEVELS[0] for k in ["uo_", "vo_", "thetao_", "so_"]
    ]
    + ["zos"],
    "3D_5": [
        k + str(j)
        for k in ["uo_", "vo_", "thetao_", "so_"]
        for j in DEPTH_I_LEVELS[:5]
    ]
    + ["zos"],
    "3D_all": [
        k + str(j)
        for k in ["uo_", "vo_", "thetao_", "so_"]
        for j in DEPTH_I_LEVELS
    ]
    + ["zos"],
    "3D_noFast_all": [
        k + str(j) for k in ["thetao_", "so_"] for j in DEPTH_I_LEVELS
    ]
    + ["zos"],
    "3D_TS_all": [k + str(j) for k in ["thetao_", "so_"] for j in DEPTH_I_LEVELS],
    "3D_onlyTemp_all": [k + str(j) for k in ["thetao_"] for j in DEPTH_I_LEVELS],
    "3D_SST_all": [
        k + str(j)
        for k in ["uo_", "vo_", "thetao_", "so_"]
        for j in DEPTH_I_LEVELS
        if not (k == "thetao_" and j == DEPTH_I_LEVELS[0])
    ]
    + ["zos"],
}
EXTRA_VARS = {
    "1": ["ur", "vr"],
    "2": ["ur", "vr", "Tm"],
    "3": ["Tm"],
    "4": ["ur", "vr", "Tm", "Tr"],
    "NoBoundary": [],
    "6": ["um", "vm"],
    "7": ["um", "vm", "Tm"],
    "8": ["um", "vm", "Tm", "Tr"],
    "9": ["ur", "vr", "tau_u", "tau_v"],
    "10": ["tau_u", "tau_v"],
    "11": ["t_ref"],
    "12": ["tau_u", "tau_v", "t_ref"],
    "13": ["ur", "vr", "Tr", "tau_u", "tau_v", "t_ref"],
    "3D": ["tauuo", "tauvo", "hfds"],
    "2D": ["tauuo", "tauvo", "hfds"],
    "3D_noFast_all": [k + str(j) for k in ["uo_", "vo_"] for j in DEPTH_I_LEVELS]
    + ["tauuo", "tauvo", "hfds"],
    "3D_5": ["tauuo", "tauvo", "hfds"],
    "3D_all": ["tauuo", "tauvo", "hfds"],
    "3D_all_hfds_anom": ["tauuo", "tauvo", "hfds", "hfds_anomalies"],
    "3D_all_hfds_anom_cuminteg": [
        "tauuo",
        "tauvo",
        "hfds",
        "hfds_anomalies",
        "cum_integrated_heat",
    ],
    "3D_all_onlyhfds_cuminteg": ["tauuo", "tauvo", "cum_integrated_heat"],
    "3D_all_SAT_tos": ["tauuo", "tauvo", "DSWRFtoa", "air_temperature_at_two_meters"],
    "3D_all_SAT": ["tauuo", "tauvo", "air_temperature_at_two_meters"],
}
OUT_VARS = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "Tm"],
    "3": ["ur", "vr"],
    "4": ["ur", "vr", "Tr"],
    "5": ["u", "v"],
    "6": ["u", "v", "T"],
    "3D": ["uo", "vo", "thetao", "so", "zos"],
    "2D": [
        k + DEPTH_I_LEVELS[0] for k in ["uo_", "vo_", "thetao_", "so_"]
    ]
    + ["zos"],
    "3D_5": [
        k + str(j)
        for k in ["uo_", "vo_", "thetao_", "so_"]
        for j in DEPTH_I_LEVELS[:5]
    ]
    + ["zos"],
    "3D_noFast_5": [
        k + str(j) for k in ["thetao_", "so_"] for j in DEPTH_I_LEVELS[:5]
    ]
    + ["zos"],
    "3D_all": [
        k + str(j)
        for k in ["uo_", "vo_", "thetao_", "so_"]
        for j in DEPTH_I_LEVELS
    ]
    + ["zos"],
    "3D_noFast_all": [
        k + str(j) for k in ["thetao_", "so_"] for j in DEPTH_I_LEVELS
    ]
    + ["zos"],
    "3D_onlyTemp_all": [k + str(j) for k in ["thetao_"] for j in DEPTH_I_LEVELS],
    "3D_onlyFast_all": [
        k + str(j) for k in ["uo_", "vo_"] for j in DEPTH_I_LEVELS
    ]
    + ["zos"],
    "3D_TS_all": [k + str(j) for k in ["thetao_", "so_"] for j in DEPTH_I_LEVELS],
    "3D_SST_all": [
        k + str(j)
        for k in ["uo_", "vo_", "thetao_", "so_"]
        for j in DEPTH_I_LEVELS
        if not (k == "thetao_" and j == DEPTH_I_LEVELS[0])
    ]
    + ["zos"],
}


def get_eval_maps(exp_num):
    # CH_3D_IDX maps the input variables to their indices in the input tensor
    # DP_3D_IDX maps the depth levels to their indices in the input tensor
    CH_3D_IDX = {}
    VAR_SET = list(dict.fromkeys(([out.split("_")[0] for out in OUT_VARS[exp_num]])))
    # assert VAR_SET[-3] == 'thetao' and VAR_SET[-2] == 'so' and VAR_SET[-1] == 'zos'
    DEPTH_SET = DEPTH_I_LEVELS
    for kt in VAR_SET:
        CH_3D_IDX[kt] = torch.tensor([])
        for i, k in enumerate(OUT_VARS[exp_num]):
            if kt in k:
                CH_3D_IDX[kt] = torch.cat([CH_3D_IDX[kt], torch.tensor([i])])
        CH_3D_IDX[kt] = CH_3D_IDX[kt].to(torch.int32)

    DP_3D_IDX = {}
    for d in DEPTH_SET:
        DP_3D_IDX[d] = torch.tensor([])
        for i, k in enumerate(OUT_VARS[exp_num]):
            if k == "zos":
                continue
            elif d == k.split("_")[-1]:
                DP_3D_IDX[d] = torch.cat([DP_3D_IDX[d], torch.tensor([i])])
        DP_3D_IDX[d] = DP_3D_IDX[d].to(torch.int32)
    if "zos" in VAR_SET:
        DP_3D_IDX[DEPTH_I_LEVELS[0]] = torch.cat(
            [DP_3D_IDX[DEPTH_I_LEVELS[0]], torch.tensor([len(OUT_VARS[exp_num]) - 1])]
        )  # zos
    DP_3D_IDX[DEPTH_I_LEVELS[0]] = DP_3D_IDX[DEPTH_I_LEVELS[0]].to(torch.int32)
    return CH_3D_IDX, DP_3D_IDX, VAR_SET, DEPTH_SET



# Region boundaries
REGIONS = {
    "Kuroshio": {"lat": [15, 41], "lon": [-215, -185]},
    "Kuroshio_Ext": {"lat": [5, 50], "lon": [-250, -175]},
    "Gulf_Stream": {"lat": [25, 50], "lon": [-70, -35]},
    "Gulf_Stream_Ext": {"lat": [27, 50], "lon": [-82, -35]},
    "Gulf_Stream_Ext2": {"lat": [26, 50.65], "lon": [-82, -50.25]},
    "Gulf_Stream_Ext3": {"lat": [26, 50.65], "lon": [-82, -34.25]},
    "Tropics": {"lat": [-5, 25], "lon": [-95, -65]},
    "Tropics_Ext": {"lat": [-5, 25], "lon": [-115, -45]},
    "South_America": {"lat": [-60, -30], "lon": [-70, -35]},
    "Africa": {"lat": [-50, -20], "lon": [5, 45]},
    "Africa_Ext": {"lat": [-55, -15], "lon": [-5, 55]},
    "Quiescent": {"lat": [-42.5, -17.5], "lon": [-155, -120]},
    "Quiescent_Ext": {"lat": [-55, -10], "lon": [-170, -110]},
    "Pacific": {"lat": [-35, 35], "lon": [-230, -80]},
    "Indian": {"lat": [-30, 28], "lon": [30, 79]},
}

GLOBAL_COMBINED_STATS = {
    "s_in": np.array(
        [
            1.19912029e-01,
            8.75121945e-02,
            1.11957607e01,
            9.65926101e-05,
            7.35161570e-05,
            2.04991480e01,
        ]
    ),
    "s_out": np.array([0.11991318, 0.08751262, 11.19576553]),
    "m_in": np.array(
        [
            -1.52130831e-03,
            4.28648579e-03,
            8.86227188e00,
            7.05813917e-06,
            2.61937937e-07,
            2.78227831e02,
        ]
    ),
    "m_out": np.array([-1.52113173e-03, 4.28606825e-03, 8.86225711e00]),
}


In [51]:
INPT_VARS["3D_noFast_all"]
EXTRA_VARS["3D_all"]
OUT_VARS["3D_noFast_all"]

['thetao_0',
 'thetao_1',
 'thetao_2',
 'thetao_3',
 'thetao_4',
 'thetao_5',
 'thetao_6',
 'thetao_7',
 'thetao_8',
 'thetao_9',
 'thetao_10',
 'thetao_11',
 'thetao_12',
 'thetao_13',
 'thetao_14',
 'thetao_15',
 'thetao_16',
 'thetao_17',
 'thetao_18',
 'so_0',
 'so_1',
 'so_2',
 'so_3',
 'so_4',
 'so_5',
 'so_6',
 'so_7',
 'so_8',
 'so_9',
 'so_10',
 'so_11',
 'so_12',
 'so_13',
 'so_14',
 'so_15',
 'so_16',
 'so_17',
 'so_18',
 'zos']

In [52]:
import xarray as xr
import fsspec

# CM4
fs_osn = fsspec.filesystem(
    "s3",
    profile="ocean_emulator",  ## This is the profile name you configured above.
)
mapper = fs_osn.get_mapper("emulators/jbusecke/ocean-emulators/CM4_5daily_v0.4.0.zarr")
ds = xr.open_zarr(mapper, consolidated=True)
hist = 1

In [53]:
OUT_VARS["3D_noFast_all"]

['thetao_0',
 'thetao_1',
 'thetao_2',
 'thetao_3',
 'thetao_4',
 'thetao_5',
 'thetao_6',
 'thetao_7',
 'thetao_8',
 'thetao_9',
 'thetao_10',
 'thetao_11',
 'thetao_12',
 'thetao_13',
 'thetao_14',
 'thetao_15',
 'thetao_16',
 'thetao_17',
 'thetao_18',
 'so_0',
 'so_1',
 'so_2',
 'so_3',
 'so_4',
 'so_5',
 'so_6',
 'so_7',
 'so_8',
 'so_9',
 'so_10',
 'so_11',
 'so_12',
 'so_13',
 'so_14',
 'so_15',
 'so_16',
 'so_17',
 'so_18',
 'zos']

In [63]:
wet_zarr = xr.open_zarr("/mnt/home/sd5313/data/CM4_5daily_v0.4.0_wetmask", consolidated=True)

In [64]:
wet_zarr

<xarray.Dataset>
Dimensions:                        (lev: 19, y: 180, x: 360)
Coordinates:
    areacello                      (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz                             (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat                            (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
  * lev                            (lev) float64 2.5 10.0 22.5 ... 5e+03 6e+03
    lon                            (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    ocean_fraction                 (lev, y, x) float64 dask.array<chunksize=(5, 90, 180), meta=np.ndarray>
    wetmask                        (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x                              (x) float64 0.5 1.5 2.5 ... 357.5 358.5 359.5
  * y                              (y) float64 -89.24 -88.25 ... 88.25 89.24
Data variables:
    __xarray_dataarray_variable__  (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
Attributes:
    __xarray_dataarray_name__:  wetmask

In [62]:
depth_ind = [var_depth_i.split("_")[-1] if var_depth_i != "zos" else "0" for var_depth_i in OUT_VARS["3D_noFast_all"]]
depths = [DEPTH_LEVELS[int(depth_i)] for depth_i in depth_ind]
wet = wet_zarr.sel(lev=depths)
wet = torch.from_numpy(wet.to_array().to_numpy().squeeze())
wet = torch.concat([wet] * (hist + 1), dim=0)
print(wet.shape)

torch.Size([78, 180, 360])


In [32]:
CH_3D_IDX, DP_3D_IDX, VAR_SET, DEPTH_SET = get_eval_maps("3D_noFast_all")
CH_3D_IDX

{'thetao': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18], dtype=torch.int32),
 'so': tensor([19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
         37], dtype=torch.int32),
 'zos': tensor([38], dtype=torch.int32)}

In [1]:
import xarray as xr

In [7]:
mean_path = "/mnt/home/sd5313/data/2024-11-12-cm4-piControl-ocean-200yr-dataset-stats/centering.nc"
std_path = "/mnt/home/sd5313/data/2024-11-12-cm4-piControl-ocean-200yr-dataset-stats/scaling-full-field.nc"

mean_ds = xr.open_dataset(mean_path)
std_ds = xr.open_dataset(std_path)

mean_ds
std_ds


<xarray.Dataset>
Dimensions:               ()
Data variables: (12/141)
    EXT                   float32 ...
    HI                    float32 ...
    HS                    float32 ...
    UI                    float32 ...
    VI                    float32 ...
    evs                   float32 ...
    ...                    ...
    vo_6                  float32 ...
    vo_7                  float32 ...
    vo_8                  float32 ...
    vo_9                  float32 ...
    wfo                   float32 ...
    zos                   float32 ...
Attributes:
    history:        Created by full-model/fv3gfs_data_process/get_stats.py. I...
    regrid_method:  conservative
    input_samples:  3651

In [2]:
# Load only specific variables
import xarray as xr
data_dir = "/mnt/home/sd5313/data/2024-11-12-cm4-piControl-ocean-200yr-monthly"
ds = xr.open_dataset(data_dir + "/0204070100.nc", engine="netcdf4")
ds


<xarray.Dataset>
Dimensions:               (time: 6, lat: 180, lon: 360)
Coordinates:
  * lat                   (lat) float64 -89.24 -88.25 -87.25 ... 88.25 89.24
  * lon                   (lon) float64 0.5 1.5 2.5 3.5 ... 357.5 358.5 359.5
  * time                  (time) object 0204-07-05 00:00:00 ... 0204-07-30 00...
Data variables: (12/141)
    EXT                   (time, lat, lon) float32 ...
    HI                    (time, lat, lon) float32 ...
    HS                    (time, lat, lon) float32 ...
    UI                    (time, lat, lon) float32 ...
    VI                    (time, lat, lon) float32 ...
    evs                   (time, lat, lon) float32 ...
    ...                    ...
    vo_6                  (time, lat, lon) float32 ...
    vo_7                  (time, lat, lon) float32 ...
    vo_8                  (time, lat, lon) float32 ...
    vo_9                  (time, lat, lon) float32 ...
    wfo                   (time, lat, lon) float32 ...
    zos                   (time, lat, lon) float32 ...
Attributes:
    history:        Dataset computed by full-model/scripts/data_process/compu...
    regrid_method:  conservative

In [ ]:
ds.to_zarr("/mnt/home/sd5313/data/CM4_5daily_v0.4.0_test.zarr", consolidated=True)

In [76]:
ds = xr.open_mfdataset(f"{data_dir}/015*.nc",
                       combine='by_coords',
                       parallel=True,
                       chunks={'time': 1},
                       engine="netcdf4")

HDF5-DIAG: Error detected in HDF5 (1.12.2) thread 26:
  #000: H5D.c line 356 in H5Dget_space(): invalid dataset identifier
    major: Invalid arguments to routine
    minor: Inappropriate type
HDF5-DIAG: Error detected in HDF5 (1.12.2) thread 27:
  #000: H5D.c line 356 in H5Dget_space(): invalid dataset identifier
    major: Invalid arguments to routine
    minor: Inappropriate type
HDF5-DIAG: Error detected in HDF5 (1.12.2) thread 28:
  #000: H5D.c line 356 in H5Dget_space(): invalid dataset identifier
    major: Invalid arguments to routine
    minor: Inappropriate type
HDF5-DIAG: Error detected in HDF5 (1.12.2) thread 29:
  #000: H5D.c line 356 in H5Dget_space(): invalid dataset identifier
    major: Invalid arguments to routine
    minor: Inappropriate type
HDF5-DIAG: Error detected in HDF5 (1.12.2) thread 30:
  #000: H5D.c line 356 in H5Dget_space(): invalid dataset identifier
    major: Invalid arguments to routine
    minor: Inappropriate type
HDF5-DIAG: Error detected in HDF5 (

RuntimeError: NetCDF: HDF error

: 

In [ ]:
ds